#  Riset analisa estimasi Recovery Aquifer JIAT (Jaringan Irigasi Air Tanah)

In [ ]:

KEDALAMAN_SUMUR = 70 
# Data_Air_Tanah = kedalaman sumur - rerata muka air tanah 
file_path = "D:/RnD_JIAT/data/Pos_AWLR_JIAT_Pondok_Kahuru.csv"

In [12]:
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots

KEDALAMAN_SUMUR = 70 

# Load the data
file_path = "D:/RnD_JIAT/data/Pos_AWLR_JIAT_Pondok_Kahuru.csv"
df_eda = pd.read_csv(file_path)

# Pastikan Waktu ke datetime
kolom_waktu = "waktu"
kolom_mat_rerata = "Muka_Air_Tanah_mean"
df_eda[kolom_waktu] = pd.to_datetime(df_eda[kolom_waktu])

# Kalkulasi Data Air Tanah
#df_eda["Data Air Tanah"] = KEDALAMAN_SUMUR - df_eda[kolom_mat_rerata]

# Membuat Dual Plot (Sumbu Y kiri dan Y kanan)
fig_dual = make_subplots(specs=[[{"secondary_y": True}]])

# Trace 1: Muka Air Tanah (TMA) - Sumbu Y Kiri
fig_dual.add_trace(
    go.Scatter(
        x=df_eda[kolom_waktu],
        y=df_eda[kolom_mat_rerata],
        name="Muka Air Tanah (TMA)",
        mode="lines",
        line=dict(color="#0ea5e9", width=2),
        fill='tozeroy', 
        fillcolor='rgba(14, 165, 233, 0.1)'
    ),
    secondary_y=False
)

# Trace 2: Data Air Tanah (Sisa Tinggi Tiang Air) - Sumbu Y Kanan
fig_dual.add_trace(
    go.Scatter(
        x=df_eda[kolom_waktu],
        y=df_eda["Data_Air_Tanah"],
        name="Data Air Tanah (Volume Riil)",
        mode="lines",
        line=dict(color="#E69F00", width=2, dash="dot")
    ),
    secondary_y=True
)

# ==============================================================================
# MENGHITUNG DINAMIS RANGE BERDASARKAN MIN/MAX DATA (+ MARGIN 10%)
# ==============================================================================
min_tma = df_eda[kolom_mat_rerata].min()
max_tma = df_eda[kolom_mat_rerata].max()

# Beri jarak ruang napas (margin offset) 10% di atas dan bawah grafik
margin_tma = (max_tma - min_tma) * 0.10
if margin_tma == 0: 
    margin_tma = 1.0 # Antisipasi jika data sensor stagnan sempurna

# Batas Y-Axis Kiri (TMA) - Dibuat terbalik: Angka besar (dalam) di bawah
tma_batas_bawah = max_tma + margin_tma 
tma_batas_atas = min_tma - margin_tma  

# Batas Y-Axis Kanan (Data Air) - Normal: Angka kecil di bawah
data_batas_bawah = KEDALAMAN_SUMUR - tma_batas_bawah
data_batas_atas = KEDALAMAN_SUMUR - tma_batas_atas

# ==============================================================================
# KOSMETIKA DASHBOARD KHAS DATA SCIENCE
# ==============================================================================
fig_dual.update_layout(
    title='<b>Analisis Cermin: Fluktuasi Turunnya Ketinggian Air Tanah (TMA vs Riil Volume)</b>',
    xaxis_title='Waktu Log Kalender',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)
# Custom Sumbu Kiri (TMA) - Warna Sky Blue (Okabe-Ito)
fig_dual.update_yaxes(
    title_text="Kedalaman Air Muka (Meter)", 
    range=[tma_batas_bawah, tma_batas_atas], # Otomatis reversed karena batas_bawah nilainya lebih besar
    secondary_y=False,
    color="#56B4E9", # Diganti ke Sky Blue
    showgrid=False
)
# Custom Sumbu Kanan (Data Air Tanah Riil Volumetrik) - Warna Jelas Orange (Okabe-Ito)
fig_dual.update_yaxes(
    title_text="Sisa Ketinggian Tiang Air (Meter)", 
    range=[data_batas_bawah, data_batas_atas], 
    secondary_y=True,
    color="#E69F00", # Diganti ke Orange Kontras
    showgrid=True,
    gridcolor='rgba(255, 255, 255, 0.1)'
)
fig_dual.show()


In [6]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import curve_fit

# Parameter Asumsi Sumur
KEDALAMAN_SUMUR = 70
file_path = "D:/RnD_JIAT/data/Pos_AWLR_JIAT_Pondok_Kahuru.csv"

# 1. Load & Preprocessing Data Dasar
df = pd.read_csv(file_path)
df['Waktu'] = pd.to_datetime(df['Waktu'])
df['Data Air Tanah'] = KEDALAMAN_SUMUR - df['Rerata Muka Air Tanah']

# 2. Filtering Data Area Kajian: 1 April - 12 April
mask_tanggal = (df['Waktu'] >= '2026-04-01') & (df['Waktu'] < '2026-04-13')
df_kajian = df.loc[mask_tanggal].copy().reset_index(drop=True)

# 3. Identifikasi Fase Recovery
# Asumsi: Kita mendemonstrasikan implementasi dengan mengambil fase recovery 
# yang dimulai dari titik volume air terendah di rentang kajian.
min_idx = df_kajian['Data Air Tanah'].idxmin()

# Ambil data sejak pompa mati (titik minimum) hingga 72 jam (3 hari) ke depan 
# sebagai fase recovery eksperimental uji coba untuk kurva regresi eksponensial.
df_recovery = df_kajian.loc[min_idx:min_idx+72].copy().reset_index(drop=True)

# 4. Membangun Model Recovery Eksponensial (Curve Fitting)
# Rumus Dasar: DAT(t) = DAT_statis - Drop * exp(-k * t)
# t adalah waktu berjalan dalam 'jam' semenjak fase recovery dimulai
df_recovery['Jam_Sejak_Mati'] = (df_recovery['Waktu'] - df_recovery['Waktu'].iloc[0]).dt.total_seconds() / 3600.0

def model_eksponensial(t, dat_statis, drop, k):
    return dat_statis - drop * np.exp(-k * t)

# Tebakan awal parameter untuk optimisasi matematika: 
# [Posisi Ideal Maks (Asimtot), Jarak Drop Kedalaman Maks, k-Laju recovery]
dat_max_tebakan = df_recovery['Data Air Tanah'].max()
drop_tebakan = dat_max_tebakan - df_recovery['Data Air Tanah'].min()
p0_tebakan = [dat_max_tebakan, drop_tebakan, 0.1]

# Eksekusi Fitting Kurva (Regresi Non-Linier)
x_data = df_recovery['Jam_Sejak_Mati'].values
y_data = df_recovery['Data Air Tanah'].values

try:
    popt, pcov = curve_fit(model_eksponensial, x_data, y_data, p0=p0_tebakan, maxfev=5000)
    dat_statis_fit, drop_fit, k_fit = popt
    
    # Rekonstruksi kurva mulus dari model matematika untuk visualisasi masa depan
    # Kita ekstrapolasikan model menebak +96 jam dari titik data terakhir
    x_model = np.linspace(0, x_data[-1] + 96, 200) 
    y_model = model_eksponensial(x_model, *popt)
    waktu_model = [df_recovery['Waktu'].iloc[0] + pd.Timedelta(hours=h) for h in x_model]
    
except Exception as e:
    print("Curve fit gagal mengkonvergen model:", e)
    dat_statis_fit, drop_fit, k_fit = p0_tebakan
    x_model, y_model, waktu_model = [], [], []

# 5. Visualisasi Hasil Kajian Data
fig = go.Figure()

# Plot Konteks Awal: Seluruh Data Kajian 1 - 12 April
fig.add_trace(go.Scatter(
    x=df_kajian['Waktu'], y=df_kajian['Data Air Tanah'],
    name="Data Rentang Area (Abu)", mode='lines', line=dict(color='gray', width=1)
))

# Plot Fokus Kajian: Titik-titik aktual fase Recovery Sensor 
fig.add_trace(go.Scatter(
    x=df_recovery['Waktu'], y=df_recovery['Data Air Tanah'],
    name="Data Ground Truth Recovery", mode='markers+lines', line=dict(color='#d946ef', width=2),
    marker=dict(size=5)
))

# Plot Matematika: Garis Regresi Model & Prediksi Ke Depan
if len(waktu_model) > 0:
    fig.add_trace(go.Scatter(
        x=waktu_model, y=y_model,
        name=f"Model Analitis Kurva (k={k_fit:.3f})", mode='lines', line=dict(color='#0ea5e9', width=3, dash='dash')
    ))
    
    # Batas Titik Jenuh Ketinggian Air Secara Model
    fig.add_hline(y=dat_statis_fit, line_dash="dot", line_color="green", 
                  annotation_text=f"Batas Maksimum (Asimtot): {dat_statis_fit:.2f} m")

fig.update_layout(
    title='<b>Prototipe Mesin ETA Recovery (Filter: 1 - 12 April 2026)</b>',
    xaxis_title='Waktu Log Kalender',
    yaxis_title='Data Sisa Tinggi Tiang Air Tanah (Meter)',
    template='plotly_dark',
    hovermode='x unified',
    legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
)

fig.show()

# 6. Hasil Analisa Konkrit
if k_fit > 0:
    # Waktu menuju 95% pemulihan = -ln((1 - 0.95)) / k = -ln(0.05) / k
    t_95 = -np.log(0.05) / k_fit
    eta_waktu = df_recovery['Waktu'].iloc[0] + pd.Timedelta(hours=t_95)
    
    print("\n" + "="*50)
    print(" " * 5 + "HASIL ANALISA MATEMATIKA RECOVERY AQUIFER")
    print("="*50)
    print(f"Ketinggian Ideal Terestimasi (Statis)	: {dat_statis_fit:.2f} Meter")
    print(f"Konstanta Laju Recovery (k)		: {k_fit:.4f} jam⁻¹")
    print(f"Tingkat Error (Residu Drop)		: {drop_fit:.2f} Meter")
    print("-" * 50)
    print(f"⏳ ETA Pemulihan Aman (95% Pulih)	: Butuh {t_95:.1f} Jam sejak mati")
    print(f"🎯 Tanggal Kelayakan Operasional	: {eta_waktu.strftime('%Y-%m-%d %H:%M')}")
    print("="*50)
# ==============================================================================
# 7. ANALISA & REKAP KONSUMSI: CHARGING vs DISCHARGING
# ==============================================================================
# Membuat kolom Delta untuk melihat Perubahan Air dari 1 jam sebelumnya
df_kajian['Delta'] = df_kajian['Data Air Tanah'].diff()

# Membuat Kolom Tanggal Khusus untuk Grouping "Dalam 1 Hari"
df_kajian['Tanggal'] = df_kajian['Waktu'].dt.date

print("\n\n" + "="*65)
print(" " * 12 + "Laporan Operasional Sumur per Hari")
print("="*65)

total_char = 0
total_dis = 0

# Iterasi/Loop Data Hari demi Hari
for tanggal, group in df_kajian.groupby('Tanggal'):
    # Menyaring baris yg grafiknya Naik (Memulih) vs Turun (Disedot)
    df_dis = group[group['Delta'] < 0] # Discharge
    df_char = group[group['Delta'] > 0] # Charge
    
    # 1 Baris = 1 Jam Data
    jam_dis = len(df_dis)
    vol_dis = abs(df_dis['Delta'].sum()) # Volume total yg diambil
    
    jam_char = len(df_char)
    vol_char = df_char['Delta'].sum()    # Volume total yg terisi lagi
    
    total_char += jam_char
    total_dis += jam_dis

    # --- LOGIKA KONVERSI WAKTU ---
    fmt_jam_char = f"{jam_char} Jam" if jam_char < 24 else f"{(jam_char/24.0):.1f} Hari"
    fmt_jam_dis  = f"{jam_dis} Jam"  if jam_dis < 24  else f"{(jam_dis/24.0):.1f} Hari"

    print(f"📅 {tanggal}")
    print(f"   🔻 Discharge (Terbuang)\t: {fmt_jam_dis} | Banyaknya: {vol_dis:.2f} Meter")
    print(f"   🔋 Charge (Pemulihan)\t: {fmt_jam_char} | Banyaknya: {vol_char:.2f} Meter")

print("-" * 65)

# Terapkan konversi hari pada laporan rekap total juga
tot_char_str = f"{total_char} Jam" if total_char < 24 else f"{(total_char/24.0):.1f} Hari"
tot_dis_str  = f"{total_dis} Jam"  if total_dis < 24  else f"{(total_dis/24.0):.1f} Hari"

print("📊 AKUMULASI KESELURUHAN (Rentang Area Kajian):")
print(f"Total Waktu Pompa Menyedot\t: {tot_dis_str}")
print(f"Total Waktu Akuifer Memulih\t: {tot_char_str}")
print("="*65)

KeyError: 'Waktu'

In [6]:
# CATATAN: Untuk menjalankan cell ini, pastikan Anda telah menginstal pustaka Prophet
# Jika belum, jalankan perintah ini di terminal env / Anaconda prompt Anda: 
# !pip install prophet

import pandas as pd
import plotly.graph_objects as go
import logging

# Menyembunyikan pesan merah / warning teknis dari modul Prophet
logging.getLogger('prophet').setLevel(logging.WARNING)

try:
    from prophet import Prophet
    has_prophet = True
except ImportError:
    print("⚠️ Modul 'prophet' belum terinstal. Grafik prediksi tidak dapat dibuat.")
    print("Silakan jalankan perintah instalasi (pip install prophet)")
    has_prophet = False

if has_prophet:
    # 1. Menyiapkan Data Tabular Khusus Prophet
    # Syarat utama: Struktur nama kolom harus 'ds' (Date/Time) dan 'y' (Nilai Skalar Target)
    df_prophet = df_kajian[['Waktu', 'Data Air Tanah']].copy()
    df_prophet.rename(columns={'Waktu': 'ds', 'Data Air Tanah': 'y'}, inplace=True)
    
    # 2. Inisiasi Sang Ahli Time-Series (Prophet)
    # interval_width=0.95 (Kepercayaan 95%) 
    # daily_seasonality=True: penting karena siklus pemompaan pasti berulang siang/malam (per 24 jam)
    model_prophet = Prophet(
        daily_seasonality=True, 
        yearly_seasonality=False, 
        weekly_seasonality=True, 
        interval_width=0.95
    )
    
    # 3. Proses Training
    model_prophet.fit(df_prophet)
    
    # 4. Ekstrapolasi / Peramalan
    # Buatkan wadah waktu kosong sejauh 4 Hari * 24 Jam = 96 Langkah Jam ke depan
    future_dataframe = model_prophet.make_future_dataframe(periods=96, freq='H')
    
    # Eksekusi Prediksi pada memori kosong tersebut
    forecast = model_prophet.predict(future_dataframe)
    
    # ==============================================================================
    # VISUALISASI HYBRID (PROPHET ML vs REGRESI FISIKA)
    # ==============================================================================
    fig_prophet = go.Figure()
    
    # Plot A: Data Asli Masa Lalu
    fig_prophet.add_trace(go.Scatter(
        x=df_prophet['ds'], y=df_prophet['y'],
        name="Data Aktual Historis", mode="lines",
        line=dict(color="silver", width=1.5)
    ))
    
    # Plot B: Pita Jangkauan Prediksi (Toleransi Error/Ketidakpastian 95%)
    fig_prophet.add_trace(go.Scatter(
        x=forecast['ds'].tolist() + forecast['ds'].tolist()[::-1],
        y=forecast['yhat_upper'].tolist() + forecast['yhat_lower'].tolist()[::-1],
        fill='toself',
        fillcolor='rgba(16, 185, 129, 0.15)', # Transparan Emerald
        line=dict(color='rgba(255,255,255,0)'),
        name="Pita Toleransi Model AI",
        hoverinfo='skip' 
    ))

    # Plot C: Garis Utama Tarikan ML Prophet
    fig_prophet.add_trace(go.Scatter(
        x=forecast['ds'], y=forecast['yhat'],
        name="Prediksi Prophet Utama", mode="lines",
        line=dict(color="#10b981", width=2.5, dash='longdash') # Warna Emerald Hijau
    ))
    
    # Menampilkan Referensi Hasil Hitungan Fisika (dari Cell Sebelumnya) sebagai Indikator Komparasi Hitungan
    if 'dat_statis_fit' in locals() and 'eta_waktu' in locals():
        
        # Sumbu Y: Asimtot Level Ideal
        fig_prophet.add_hline(
            y=dat_statis_fit, line_dash="dot", line_color="#f59e0b", line_width=2,
            annotation_text=f"Batas Ideal CurveFit: {dat_statis_fit:.1f} m"
        )
        
        # Sumbu X: Momentum Waktu Sembuh (Digambar SETENGAH-SETENGAH)
        # 1. Gambar Garis Vertikalnya (Tanpa Teks agar Plotly tidak memproses matematika teks)
        fig_prophet.add_vline(
            x=eta_waktu, line_dash="dashdot", line_color="#f59e0b", line_width=2
        )
        
        # 2. Gambar Teks Anotasinya secara Terpisah (Mengambang di paling atas kanvas 'yref=paper')
        fig_prophet.add_annotation(
            x=eta_waktu, y=1.0, yref="paper", 
            text="Moment ETA CurveFit",
            showarrow=False, font=dict(color="#f59e0b"), 
            xanchor="left", xshift=5
        )
        
    fig_prophet.update_layout(
        title='<b>Cross Validation: Machine Learning (FB Prophet) Vs Rumus Fisika</b>',
        xaxis_title='Waktu Log Kalender (Masa Lalu -> Esok Hari)',
        yaxis_title='Sisa Tinggi Tiang Air Tanah (Meter)',
        template='plotly_dark',
        hovermode='x unified',
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
    )
    
    fig_prophet.show()

10:17:54 - cmdstanpy - INFO - Chain [1] start processing
10:17:54 - cmdstanpy - INFO - Chain [1] done processing
c:\Users\fadel\anaconda3\envs\torch\Lib\site-packages\prophet\forecaster.py:1872: FutureWarning:

'H' is deprecated and will be removed in a future version, please use 'h' instead.

c:\Users\fadel\anaconda3\envs\torch\Lib\site-packages\plotly\io\_json.py:560: UserWarning:

Discarding nonzero nanoseconds in conversion.



In [10]:
import pandas as pd
import numpy as np

file_path = "D:/RnD_JIAT/data/Pos_AWLR_JIAT_Pondok_Kahuru.csv"
KEDALAMAN_SUMUR = 70

df = pd.read_csv(file_path)
df['Waktu'] = pd.to_datetime(df['Waktu'])
df['Data Air Tanah'] = KEDALAMAN_SUMUR - df['Rerata Muka Air Tanah']

mask_tanggal = (df['Waktu'] >= '2026-04-01') & (df['Waktu'] < '2026-04-13')
df_kajian = df.loc[mask_tanggal].copy().reset_index(drop=True)

df_kajian['Delta'] = df_kajian['Data Air Tanah'].diff()

total_rows = len(df_kajian)
total_hours = (df_kajian['Waktu'].max() - df_kajian['Waktu'].min()).total_seconds() / 3600 + 1

n_charge = len(df_kajian[df_kajian['Delta'] > 0])
n_discharge = len(df_kajian[df_kajian['Delta'] < 0])
n_stagnant = len(df_kajian[df_kajian['Delta'] == 0])
n_nan = df_kajian['Delta'].isna().sum()

print("=== EDA MENDALAM VALIDASI LOGIKA CHARGE/DISCHARGE ===")
print(f"Total Baris Data Aktual: {total_rows}")
print(f"Total Jam Seharusnya (Time Range): {total_hours}")
print(f"Status Baris:")
print(f"  - Charging (Delta > 0)    : {n_charge} Jam")
print(f"  - Discharging (Delta < 0) : {n_discharge} Jam")
print(f"  - Stagnant (Delta == 0)   : {n_stagnant} Jam")
print(f"  - Baris Pertama (NaN)     : {n_nan} Jam")

print(f"\nDischarge + Charge + Stagnant + NaN = {n_charge + n_discharge + n_stagnant + n_nan} (Sama dengan Total baris? {total_rows == (n_charge + n_discharge + n_stagnant + n_nan)})")

# Let's check max consecutive stagnant hours
stagnant_groups = (df_kajian['Delta'] != 0).cumsum()
stagnant_counts = df_kajian[df_kajian['Delta'] == 0].groupby(stagnant_groups).size()
max_stag = stagnant_counts.max() if not stagnant_counts.empty else 0
print(f"Max Jam Stagnan Beruntun: {max_stag} Jam")


=== EDA MENDALAM VALIDASI LOGIKA CHARGE/DISCHARGE ===
Total Baris Data Aktual: 288
Total Jam Seharusnya (Time Range): 288.0
Status Baris:
  - Charging (Delta > 0)    : 183 Jam
  - Discharging (Delta < 0) : 93 Jam
  - Stagnant (Delta == 0)   : 11 Jam
  - Baris Pertama (NaN)     : 1 Jam

Discharge + Charge + Stagnant + NaN = 288 (Sama dengan Total baris? True)
Max Jam Stagnan Beruntun: 2 Jam


## Request Fetch dari swagger

In [3]:
import json
import pandas as pd
from datetime import datetime, timedelta
import os
from tqdm.notebook import tqdm  # -> Impor spesifik bawaan Jupyter

# Lokasi file path
json_path = "d:/RnD_JIAT/data/jiat_info.json"
output_dir = "d:/RnD_JIAT/data"

with open(json_path, 'r') as f:
    jiat_info = json.load(f)

# Rentang Tanggal
tgl_awal = datetime(2026, 1, 1)
tgl_akhir = datetime(2026, 4, 13)

# Ubah generator ke bentuk list biasa agar Tqdm bisa mendeteksinya
daftar_hari = [tgl_awal + timedelta(n) for n in range(int((tgl_akhir - tgl_awal).days) + 1)]

print("⚡ INITIATING BATCH DOWNLOAD API ⚡\n")

for pos_name, metadata in jiat_info.items():
    id_logger = str(metadata["id_logger"])
    print(f"🌊 [Pos: {pos_name} | ID: {id_logger}] -> Proses download dimulai...")
    
    bucket_data_pos = []

    # BUNGKUS iterasi daftar hari dengan TQDM beserta nama Label Bar-nya (desc)
    for tgl_satuan in tqdm(daftar_hari, desc=f"Loading {pos_name}", leave=True):
        str_tgl = tgl_satuan.strftime("%Y-%m-%d")
        
        # Panggil fetch function per satuan hari
        df_harian = fetch_data_by_logger(id_logger, str_tgl, str_tgl)
        
        if not df_harian.empty:
            bucket_data_pos.append(df_harian)
            
    # Selesaikan Penyatuan data dan Saving
    if bucket_data_pos:
        df_full = pd.concat(bucket_data_pos, ignore_index=True)
        safe_filename = pos_name.replace(" ", "_") + ".csv"
        save_path = os.path.join(output_dir, safe_filename)
        
        df_full.to_csv(save_path, index=False)
        print(f"💾 SUKSES -> Tersimpan {len(df_full)} total baris ke '{safe_filename}'\n")
    else:
        print(f"⚠️ {pos_name} -> Kosong / Nihil.\n")
        
print("🎉 BATCH SELESAI!")

⚡ INITIATING BATCH DOWNLOAD API ⚡

🌊 [Pos: Pos AWLR JIAT Curug Agung | ID: 10360] -> Proses download dimulai...


Loading Pos AWLR JIAT Curug Agung:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug

Loading Pos AWLR JIAT Sanding:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 23 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 19 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80.0m
✅ Berhasil menarik & menyaring 24 baris data.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sanding | Logger: 10361 | Sumur: 80

...

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀

Loading Pos AWLR JIAT Kamasan:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamasan | Logger: 10364 | Sumur: 100.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Kamas

Loading Pos AWLR JIAT Sindang Mandi:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Sindang Mandi | Logger: 10365 | Sumur: 70.0m
⚠️ Data kosong pada API untuk interval waktu ini.
✅ Ambil Informasi dari jiat_info.json 
🚀

In [12]:
import json
import requests

# 1. Baca file jiat_info.json
json_filepath = "d:/RnD_JIAT/data/jiat_info.json"

with open(json_filepath, 'r') as file:
    jiat_data = json.load(file)

# 2. Ambil id_logger dan id_parameter dari salah satu pos (misal: Pondok Kahuru)
pos_name = "Pos AWLR JIAT Pondok Kahuru"
id_logger = jiat_data[pos_name]["id_logger"]
id_parameter = jiat_data[pos_name]["id_parameter"]

print(f"Mencoba fetch {pos_name} | Logger: {id_logger} | Parameter: {id_parameter}")

# 3. Request API sederhana
url = f"https://mini-stesy.beacontelemetry.com/api/v1/loggers/{id_logger}/hourly-range"

# Parameter query dari swagger
params = {
    "parameter_id": id_parameter,
    "start_date": "2026-01-01", # Silakan sesuaikan tanggal
    "end_date": "2026-01-31"
}

try:
    response = requests.get(url, params=params)
    response.raise_for_status() # Menghasilkan warning kalau ada error HTTP (misal 404, 500)
    
    # Parse output ke Json
    data_api = response.json()
    
    print("Fetch Berhasil! Menampilkan 2 data pertama:")
    # Kalau data endpoint berbentuk list, kita print 2 teratas
    print(data_api[:2] if isinstance(data_api, list) else data_api)
    
except requests.exceptions.RequestException as e:
    print(f"Gagal melakukan request API: {e}")


Mencoba fetch Pos AWLR JIAT Pondok Kahuru | Logger: 10363 | Parameter: 28
Fetch Berhasil! Menampilkan 2 data pertama:
{'status': True, 'message': 'Data per jam berhasil diambil', 'logger': {'id_logger': '10363', 'nama_logger': 'AWLR JIAT Pondok Kahuru', 'nama_lokasi': 'Pos AWLR JIAT Pondok Kahuru', 'tabel_main': 'awlr'}, 'parameter': {'id_param': 28, 'nama_parameter': 'Muka_Air_Tanah', 'kolom_sensor': 'sensor1', 'satuan': 'm', 'tipe_graf': 'spline', 'metode_agregasi': 'avg'}, 'range': {'start_date': '2026-01-01', 'end_date': '2026-01-31'}, 'data': [{'waktu': '2026-01-19 10:00:00', 'nilai': 0.78, 'min': 0.78, 'max': 0.78}, {'waktu': '2026-01-19 11:00:00', 'nilai': 1.88, 'min': 1.87, 'max': 1.88}, {'waktu': '2026-01-19 12:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9}, {'waktu': '2026-01-19 13:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9}, {'waktu': '2026-01-19 14:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9}, {'waktu': '2026-01-19 15:00:00', 'nilai': 1.91, 'min': 1.91, 'max': 1.91}, {'wa

In [16]:
jiat_data[pos_name]["Kedalaman Sumur"]

70.0

In [23]:
data_api.get("data")
# ambil data
# 'waktu': '2026-01-19 11:00:00', 'nilai': 1.88,
# jiat_data[pos_name]["Kedalaman Sumur"]
# data air tanah : Kedalaman Sumur - nilai

[{'waktu': '2026-01-19 10:00:00', 'nilai': 0.78, 'min': 0.78, 'max': 0.78},
 {'waktu': '2026-01-19 11:00:00', 'nilai': 1.88, 'min': 1.87, 'max': 1.88},
 {'waktu': '2026-01-19 12:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9},
 {'waktu': '2026-01-19 13:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9},
 {'waktu': '2026-01-19 14:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9},
 {'waktu': '2026-01-19 15:00:00', 'nilai': 1.91, 'min': 1.91, 'max': 1.91},
 {'waktu': '2026-01-19 17:00:00', 'nilai': 1.91, 'min': 1.91, 'max': 1.91},
 {'waktu': '2026-01-19 18:00:00', 'nilai': 1.91, 'min': 1.91, 'max': 1.91},
 {'waktu': '2026-01-19 19:00:00', 'nilai': 1.91, 'min': 1.91, 'max': 1.91},
 {'waktu': '2026-01-19 20:00:00', 'nilai': 1.91, 'min': 1.9, 'max': 1.91},
 {'waktu': '2026-01-19 21:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.91},
 {'waktu': '2026-01-19 22:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.91},
 {'waktu': '2026-01-19 23:00:00', 'nilai': 1.9, 'min': 1.9, 'max': 1.9},
 {'waktu': '2026-01-20 00:00:

In [39]:
import json
import requests
import pandas as pd

def fetch_data_by_logger(id_logger, start_date, end_date):
    """
    Fungsi fetch API dari jiat_info.json, mengubah nilai Muka Air Tanah per Kolom
    dan menstrukturisasi ulang (Rename & Pilih Kolom Tertentu).
    """
    json_filepath = "d:/RnD_JIAT/data/jiat_info.json"

    # 1. Baca konfigurasi JSON
    with open(json_filepath, 'r') as file:
        print("✅ Ambil Informasi dari jiat_info.json ")
        jiat_data = json.load(file)

    # 2. Cari id_parameter, Kedalaman Sumur, dan nama Pos berdasarkan id_logger
    id_parameter = None
    kedalaman_sumur = None
    pos_name = None
    
    for nama_pos, metadata in jiat_data.items():
        if str(metadata.get("id_logger")) == str(id_logger):
            id_parameter = metadata.get("id_parameter")
            kedalaman_sumur = metadata.get("Kedalaman Sumur") 
            pos_name = nama_pos
            break
            
    if id_parameter is None:
        print(f"❌ Error: Logger ID '{id_logger}' tidak ditemukan di jiat_info.json")
        return pd.DataFrame()

    print(f"🚀 Memproses {pos_name} | Logger: {id_logger} | Sumur: {kedalaman_sumur}m")

    # 3. Request API 
    url = f"https://mini-stesy.beacontelemetry.com/api/v1/loggers/{id_logger}/hourly-range"
    params = {
        "parameter_id": id_parameter,
        "start_date": start_date,
        "end_date": end_date
    }

    try:
        response = requests.get(url, params=params)
        response.raise_for_status() 
        
        # Parse JSON ke dict bawaan Python
        data_api = response.json() 
        
        # 4. Ambil isi sub-array-nya langsung dari dict
        all_data = data_api.get("data", [])
            
        if not all_data:
             print("⚠️ Data kosong pada API untuk interval waktu ini.")
             return pd.DataFrame()
        
        # 5. Konversi Object menjadi DataFrame
        df = pd.DataFrame(all_data) 
        
        # Pastikan tipe/format kolom waktu rapi
        df['waktu'] = pd.to_datetime(df['waktu'])
        df = df.sort_values('waktu').reset_index(drop=True)
        
        # Kalkulasi Fisik ke Data Air Tanah
        df['Kedalaman Sumur'] = kedalaman_sumur
        df['Data_Air_Tanah'] = df['Kedalaman Sumur'] - df['nilai']
        
        # ------------------------------------------------------------
        # 6. MENGATUR STRUKTUR & RENAME KOLOM SECARA SPESIFIK
        # ------------------------------------------------------------
        # (A) Hanya menyisakan list kolom yang Anda minta:
        list_col = ["waktu", "nilai", "Kedalaman Sumur", "Data_Air_Tanah"]
        
        # Ekstrak dataframe tersebut (gunakan copy() agar modifikasi di memori baru aman)
        df_bersih = df[list_col].copy()
        
        # (B) Rename nama "nilai" dan "Kedalaman Sumur"
        df_bersih = df_bersih.rename(columns={
            "nilai": "Muka_Air_Tanah_mean",
            "Kedalaman Sumur": "Kedalaman_Sumur"
        })
        
        print(f"✅ Berhasil menarik & menyaring {len(df_bersih)} baris data.")
        return df_bersih
        
    except requests.exceptions.RequestException as e:
        print(f"❌ Gagal melakukan request API: {e}")
        return pd.DataFrame()


# ==========================================
# CONTOH CARA PENGGUNAAN
# ==========================================

# Tarik data (contoh Pondok Kahuru '10363' atau Curug Agung '10360')
dt_api = fetch_data_by_logger("10360", "2026-01-01", "2026-01-31")

# Lihat hasilnya (sudah difilter & diganti nama headernya)
if not dt_api.empty:
    display(dt_api.head())


✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 643 baris data.


,waktu,Muka_Air_Tanah_mean,Kedalaman_Sumur,Data_Air_Tanah
0,2026-01-01 00:00:00,6.96,100.0,93.04
1,2026-01-01 01:00:00,6.96,100.0,93.04
2,2026-01-01 02:00:00,6.96,100.0,93.04
3,2026-01-01 03:00:00,6.96,100.0,93.04
4,2026-01-01 04:00:00,6.96,100.0,93.04


In [ ]:
import sys
import os

# Maju satu folder ke atas (kembali ke root direktori D:/RnD_JIAT)
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))
# Tambahkan ke path agar Python bisa mendeteksi folder "data/"
if parent_dir not in sys.path:
    sys.path.append(parent_dir)
# Jalankan import-nya

from data.get_data import fetch_data_by_logger

dt_api = fetch_data_by_logger("10360", "2026-01-01", "2026-01-31")

# Lihat hasilnya (sudah difilter & diganti nama headernya)
if not dt_api.empty:
    display(dt_api.head())

✅ Ambil Informasi dari jiat_info.json 
🚀 Memproses Pos AWLR JIAT Curug Agung | Logger: 10360 | Sumur: 100.0m
✅ Berhasil menarik & menyaring 643 baris data.


,waktu,Muka_Air_Tanah_mean,Kedalaman_Sumur,Data_Air_Tanah
0,2026-01-01 00:00:00,6.96,100.0,93.04
1,2026-01-01 01:00:00,6.96,100.0,93.04
2,2026-01-01 02:00:00,6.96,100.0,93.04
3,2026-01-01 03:00:00,6.96,100.0,93.04
4,2026-01-01 04:00:00,6.96,100.0,93.04


In [ ]:
dt_api = fetch_data_by_logger("10360", "2026-01-01", "2026-01-31")

# Lihat hasilnya (sudah difilter & diganti nama headernya)
if not dt_api.empty:
    display(dt_api.head())


In [ ]:
df_hasil[""]

,waktu,nilai,min,max,Kedalaman Sumur,Data_Air_Tanah
0,2026-01-01 00:00:00,6.96,6.96,6.97,100.0,93.04
1,2026-01-01 01:00:00,6.96,6.96,6.97,100.0,93.04
2,2026-01-01 02:00:00,6.96,6.96,6.97,100.0,93.04
3,2026-01-01 03:00:00,6.96,6.96,6.97,100.0,93.04
4,2026-01-01 04:00:00,6.96,6.96,6.97,100.0,93.04
...,...,...,...,...,...,...
638,2026-01-31 19:00:00,7.52,7.52,7.53,100.0,92.48
639,2026-01-31 20:00:00,7.52,7.52,7.53,100.0,92.48
640,2026-01-31 21:00:00,7.52,7.52,7.53,100.0,92.48
641,2026-01-31 22:00:00,7.51,7.52,7.52,100.0,92.49
